# Implicit ABBA 2 simultaneous-projection symplecticity study

This versioned experiment evaluates ImplicitABBA2, the second-order endpoint-time ABBA map closed by Hairer's symmetric projection through the simultaneous output--multiplier formulation. For one guiding-centre particle, every Newton correction solves equation (21) from docs/tex/ABBA_implicit_2/ABBA_implicit_2.tex:

$$
\begin{pmatrix}
I_4 & -[I_4+D\Phi^{\mathrm{ABBA}}]G^T\\
G & 0
\end{pmatrix}
\begin{pmatrix}\Delta\mathcal Y_{n+1}\\\Delta\mu\end{pmatrix}
=-
\begin{pmatrix}d(\mathcal Y_{n+1},\mu)\\g(\mathcal Y_{n+1})\end{pmatrix}.
$$

The $6\times6$ system is assembled independently for every particle. Its Schur complement is the reduced multiplier Jacobian, so an exact converged root defines the same symmetric and symplectic physical map as implicit ABBA 1.

For every complete finite-tolerance step, this study differentiates the emitted physical map by centered finite differences. If $J_n$ is one step Jacobian and $DG_n$ is the accumulated Jacobian from the initial condition, the recorded defects are

$$
\varepsilon_{\mathrm{local},n}=\frac{\|J_n^T\Omega J_n-\Omega\|_F}{\|\Omega\|_F},\qquad
\varepsilon_{\mathrm{flow},n}=\frac{\|DG_n^T\Omega DG_n-\Omega\|_F}{\|\Omega\|_F}.
$$

Transported-polygon area, determinant drift, Newton residuals, iteration counts, and projection multipliers provide complementary geometric and nonlinear-solver diagnostics.

API migration: this notebook uses the current simulation API. Physical rho and eta belong to dynamics or study settings; initial configurations store geometry. Stored outputs were cleared and should be regenerated before scientific interpretation.


In [ ]:
%matplotlib inline
import numpy as np

from studies import (
    ImplicitABBASymplecticityConfig,
    RandomPotentialConfig,
    centered_circle,
    pi_area_steps,
    run_abba2_simultaneous_state_multiplier_symplecticity_study,
)
from visualization import display_animation


## Reproducible configuration

The potential spectrum and seed, interpolation order, boundary geometry, guiding-centre radius, integration span, three ABBA steps, observation interval, finite-difference scale, and Newton controls are explicit below. The physical configuration and differentiation scale match the other ABBA and RK4 symplecticity experiments in this directory.

In [ ]:
potential_config = RandomPotentialConfig(
    amplitude=0.7,
    max_wave_number=25,
    nx=64,
    ny=64,
    seed=27,
    interpolation_order=5,
)
potential = potential_config.build()

circle_radius = 0.5
circle_points = 16
rho = 0.3
circle = centered_circle(
    potential,
    radius=circle_radius,
    points=circle_points,
    
)

finite_difference_relative_step = float(np.cbrt(np.finfo(float).eps))
study_config = ImplicitABBASymplecticityConfig(
    steps=pi_area_steps(40, 80, 160),
    t_span=(0.0, 4 * np.pi),
    save_interval=np.pi / 8,
    rho=rho,
    chunk_size=16,
    progress=True,
    block_prefix="implicit_abba_2_symplecticity",
    finite_difference_relative_step=finite_difference_relative_step,
    newton_absolute_tolerance=1e-13,
    newton_relative_tolerance=1e-12,
    newton_max_iterations=12,
)

print(potential_config)
print(study_config)
print(
    f"Circle: {circle_points} points; t={study_config.t_span}; "
    f"{study_config.output_sample_count} saved states; "
    f"finite-difference relative step={finite_difference_relative_step:.8e}"
)


## Implicit ABBA 2 integrations and persisted Jacobians

Newton starts from $\mu_0=0$ and $\mathcal Y_{n+1}^{(0)}=\Phi^{\mathrm{ABBA}}(\mathcal Y_n)$ on every step. It monitors the infinity norm of the combined defects $(d,g)$ and reevaluates the exact simultaneous Jacobian after each correction. Every main-grid ABBA step contributes to the accumulated centered-difference Jacobian. Scalar diagnostics and full local and accumulated matrices are persisted at the common observation times.

In [ ]:
result = run_abba2_simultaneous_state_multiplier_symplecticity_study(
    potential,
    circle,
    notebook_path=(
        "notebooks/experiments/symplecticity/"
        "gc_area_and_implicit_2_abba_symplecticity.ipynb"
    ),
    config=study_config,
    metadata={
        **potential_config.metadata(),
        "circle_radius": circle_radius,
        "circle_points": circle_points,
        "study_kind": "versioned_implicit_2_symplecticity_experiment",
    },
)
result.print_summary()


## Simultaneous-solver audit

The experiment must identify equation (21) as the formulation used by every integration. This check prevents the notebook from silently evaluating the reduced compatibility method. Residual arrays must also be finite, non-negative, and aligned with the number of main integration steps.

In [ ]:
for step in study_config.steps:
    solution = result.solutions[step.label]
    diagnostics = solution.diagnostics
    assert diagnostics["projection_formulation"] == (
        "simultaneous_state_multiplier"
    )
    iterations = np.asarray(diagnostics["newton_iterations"])
    residuals = np.asarray(diagnostics["newton_residual_norms"])
    multipliers = np.asarray(diagnostics["projection_multiplier_norms"])
    assert iterations.shape == residuals.shape == multipliers.shape
    assert iterations.size == diagnostics["step_count"]
    assert np.all(np.isfinite(residuals)) and np.all(residuals >= 0.0)
    print(
        f"{step.label}: steps={iterations.size}, "
        f"max iterations={int(np.max(iterations))}, "
        f"max simultaneous residual={float(np.max(residuals)):.8e}"
    )


## Time-dependent structural diagnostics

The local matrix defect tests each simultaneous projected step independently. The accumulated defect and determinant error test the complete discrete flow from the initial boundary. The area panel reports the transported polygon and must be interpreted separately because straight polygon edges do not reproduce the curved image of the continuous boundary exactly.

In [ ]:
diagnostic_figure, diagnostic_axes = result.plot_diagnostics()


## Measured symplecticity floor

An exact-root symmetric projection is structurally symplectic, but this experiment differentiates the emitted finite-tolerance control flow. Once the local defect is controlled by centered-difference, Newton, and floating-point errors, no temporal convergence order should be fitted. The log--log plot therefore compares the measured numerical floor across integration steps.

In [ ]:
defect_floor_figure, defect_floor_axis = result.plot_defect_floor()


## Nonlinear projection diagnostics

Iteration and residual histories verify the simultaneous solve. The maximum multiplier is expected to decrease approximately as $O(h^3)$; this is a consistency property of the symmetric projection and not the global trajectory order.

In [ ]:
solver_figure, solver_axes = result.plot_solver_diagnostics()


## Comparative animation

The animation synchronizes the effective potential, transported implicit ABBA 2 contours, relative polygon-area error, and accumulated physical-flow symplecticity defect. The same color identifies each integration step in every panel.

In [ ]:
display_animation(
    result.animate(
        frames=None,
        interval=120,
    )
)


## Interpretation

This experiment evaluates the actual finite-tolerance ImplicitABBA2 map, not merely the ideal symbolic root. Small local defects support the symplectic structure of the simultaneous formulation; accumulated defects expose round-off and differentiation errors propagated over the complete run. Newton residuals and formulation metadata verify equation (21), while the multiplier study checks the expected cubic small-step scaling. Polygonal area error remains a separate finite-boundary diagnostic and must not be identified with the matrix symplecticity defect.